# B06-01 Clean v2 — 원시 성장곡선과 배지 조건의 연결 감사

**대상:** Aida–Ying 공개 *E. coli* OD 성장 데이터 (Figshare 28342064). **선행:** B06-00 자료 9개 확보.

이번 v2는 v1의 파일·시간축·품질 필드 검사를 유지하면서 `CurveNNNNN` ↔ `Label` ↔ `Assay ID` ↔ `Condition ID` 대응을 **라운드 단위로 정밀 검사**합니다. 특히 Round 03의 `Curve` 범위 겹침과 전역 중복을 실제 데이터로 판별합니다.

**원칙:** 단순히 행 순서가 같다고 연결하지 않습니다. 검증을 통과하지 않으면 과학 분석용 결합 데이터는 생성하지 않습니다. 서로 다른 실험 라운드에서 같은 Curve 이름이 등장하면 **라운드별 복합 키**가 필요할 수 있습니다.

출처: https://figshare.com/articles/dataset/Bacterial_growth_profiles_across_one-thousand_chemical-defined_media/28342064

## 1. 실행 환경 및 파일 확인
Notebook을 프로젝트 루트 또는 `notebooks` 폴더에서 실행하세요. B06-00의 9개 Excel 파일을 재다운로드하지 않습니다.

In [ ]:
# 1. 경로·패키지 확인: 변경된 프로젝트 구조에 대응
from pathlib import Path
import re, json, sys, hashlib
from collections import Counter
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
roots = [cwd, cwd.parent, cwd / 'bioimage-ai']
PROJECT_ROOT = next((r for r in roots if (r/'data'/'bioculture'/'media_aida_ying_2025').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트(data/bioculture/media_aida_ying_2025)를 확인하세요.')
DATA_DIR = PROJECT_ROOT/'data'/'bioculture'/'media_aida_ying_2025'
OUT_DIR = PROJECT_ROOT/'results'/'bioculture'/'B06_01_link_audit_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ROUND_FILES = sorted(DATA_DIR.glob('BW25113_Growth_Round*.xlsx'))
MEDIUM_FILE = DATA_DIR/'BW25113_Medium composition.xlsx'
EVAL_FILE = DATA_DIR/'BW25113_GrowthDataEvaluation.xlsx'
assert len(ROUND_FILES)==7 and MEDIUM_FILE.is_file() and EVAL_FILE.is_file(), '입력 9개 파일 누락'
print('Python:',sys.executable)
print('Data:',DATA_DIR)
print('Outputs:', OUT_DIR)

## 2. 원시 자료의 구조 확인
`Label`, `Assay ID`, `Condition ID`를 **문자열**로 읽어 앞자리 0과 라운드 표기를 보존합니다. 원본 표는 수정하지 않습니다.

In [ ]:
# 2. Excel 원본 읽기와 파일별 개괄 요약
medium = pd.read_excel(MEDIUM_FILE, engine='openpyxl', dtype={'Label':'string','Assay ID':'string','Condition ID':'string'})
evaluation = pd.read_excel(EVAL_FILE, engine='openpyxl', dtype={'Label':'string','Assay ID':'string','Condition ID':'string'})
rounds = {}
for path in ROUND_FILES:
    rid=int(re.search(r'Round(\d+)',path.name,re.I).group(1))
    rounds[rid] = pd.read_excel(path,engine='openpyxl')
frames = [('medium',medium),('evaluation',evaluation)] + [(f'round_{r:02d}',f) for r,f in sorted(rounds.items())]
file_schema = pd.DataFrame([{'file':name,'rows':len(f),'columns':len(f.columns),'first_columns':' | '.join(map(str,f.columns[:8]))} for name,f in frames])
display(file_schema)
file_schema.to_csv(OUT_DIR/'file_schema.csv',index=False,encoding='utf-8-sig')

## 3. 곡선 인벤토리·시간축·전역 중복
라운드별 `Curve` 열뿐 아니라 전체 7개 파일을 합쳐 **중복된 Label**을 검사합니다. 특히 Round 03과 Round 04의 교집합을 별도로 출력합니다.

In [ ]:
# 3. 곡선 전체 목록, 라운드별 구조, 번호 공백/중복 검사
curve_rows=[]; round_rows=[]
expected={1:2628,2:2640,3:960,4:2640,5:2640,6:1320,7:780}
for rid,df in sorted(rounds.items()):
    time_col=next((c for c in df.columns if str(c).strip().lower()=='time (h)'),None)
    assert time_col is not None, f'Round {rid}: 시간 열 없음'
    curves=[str(c) for c in df.columns if re.fullmatch(r'Curve\d+',str(c),re.I)]
    ts=pd.to_numeric(df[time_col],errors='coerce')
    numeric=df[curves].apply(pd.to_numeric,errors='coerce')
    nums=[int(re.search(r'\d+',x).group()) for x in curves]
    round_rows.append({'round':rid,'expected_count':expected[rid],'observed_count':len(curves),
         'n_time':len(ts),'time_min_h':ts.min(),'time_max_h':ts.max(),
         'time_strictly_increasing':bool(ts.notna().all() and (ts.diff().dropna()>0).all()),
         'curve_min':min(nums),'curve_max':max(nums),'unique_curve_names':len(set(curves)),
         'missing_fraction':float(numeric.isna().mean().mean())})
    for curve in curves:
        curve_rows.append({'round':rid,'Label':curve,'curve_number':int(re.search(r'\d+',curve).group()),
                           'n_valid_timepoints':int(numeric[curve].notna().sum())})
curve_inventory=pd.DataFrame(curve_rows)
round_audit=pd.DataFrame(round_rows)
duplicates=curve_inventory.loc[curve_inventory.duplicated('Label',keep=False)].sort_values(['Label','round'])
round3=set(curve_inventory.loc[curve_inventory['round']==3,'Label'])
round4=set(curve_inventory.loc[curve_inventory['round']==4,'Label'])
print('Total curves:',len(curve_inventory),'Unique labels:',curve_inventory['Label'].nunique())
print('Global duplicate occurrences:',len(duplicates),'Unique duplicated labels:',duplicates['Label'].nunique())
print('Round03 ∩ Round04:',len(round3&round4),'sample:',sorted(round3&round4)[:12])
display(round_audit)
display(duplicates.head(25))
round_audit.to_csv(OUT_DIR/'round_time_audit.csv',index=False,encoding='utf-8-sig')
curve_inventory.to_csv(OUT_DIR/'curve_inventory.csv',index=False,encoding='utf-8-sig')
duplicates.to_csv(OUT_DIR/'duplicate_curve_labels.csv',index=False,encoding='utf-8-sig')

## 4. 메타데이터의 기본 키와 Round 정보
메타데이터 `Assay ID`에서 `RoundXX_YYYY`를 파싱합니다. `Label`의 완전성·유일성, `Assay ID`와 조건 ID의 관계를 검증하고, **평가표와 조성표가 동일한 연결을 기록하는지** 점검합니다.

In [ ]:
# 4. 문자열 키의 유일성·누락·메타데이터 사이의 1:1 매칭
ids=['Label','Assay ID','Condition ID']
for name,f in [('medium',medium),('evaluation',evaluation)]:
    assert set(ids).issubset(f.columns), f'{name} 식별 열 누락'
    for c in ids:
        print(name,c,'missing',int(f[c].isna().sum()),'unique',f[c].nunique(), 'duplicates',int(f[c].duplicated().sum()))
    print(name,'sample:',f[ids].head(3).to_dict('records'))

meta_merge=evaluation[ids].merge(medium[ids],on='Label',how='outer',indicator=True,
    suffixes=('_evaluation','_medium'),validate='one_to_one')
matched=meta_merge['_merge'].eq('both')
meta_assay_equal=bool((meta_merge.loc[matched,'Assay ID_evaluation']==meta_merge.loc[matched,'Assay ID_medium']).all())
meta_cond_equal=bool((meta_merge.loc[matched,'Condition ID_evaluation']==meta_merge.loc[matched,'Condition ID_medium']).all())
print('Metadata label sets equal:',bool(matched.all()))
print('Assay ID agreement:',meta_assay_equal,'Condition ID agreement:',meta_cond_equal)

assay_pattern=r'^Round(\d+)_(\d+)$'
for f in (medium,evaluation):
    parts=f['Assay ID'].str.extract(assay_pattern)
    f['_assay_round']=pd.to_numeric(parts[0],errors='coerce').astype('Int64')
    f['_assay_local']=pd.to_numeric(parts[1],errors='coerce').astype('Int64')
print('Unparseable assay IDs in medium:',int(medium['_assay_round'].isna().sum()))
print('Medium assay rounds:',medium['_assay_round'].value_counts(dropna=False).sort_index().to_dict())

## 5. **가장 중요한 검사:** 원본 Curve Label과 메타데이터 연결
원본 성장곡선의 `(Round, Label)`을 조성표의 `(_assay_round, Label)`과 대조합니다. ① 해당 Round에 후보가 없는 경우 ② 둘 이상인 경우 ③ 유일하게 연결되는 경우를 나눠 기록합니다. **여기서는 Label을 억지로 숫자로 바꾸지 않습니다.**

In [ ]:
# 5. Round+Label 복합 키로 후보 매칭 (진단용; 최종 통합 분석표는 만들지 않음)
curve_keys=curve_inventory[['round','Label']].copy()
medium_keys=medium[['Label','Assay ID','Condition ID','_assay_round','_assay_local']].copy()
medium_keys=medium_keys.rename(columns={'_assay_round':'round'})
# 원본 곡선마다 동일 Round+Label을 가진 메타데이터가 몇 행인지 기록
candidate_counts=(medium_keys.groupby(['round','Label'],dropna=False).size()
                  .rename('metadata_candidates').reset_index())
link_qc=curve_keys.merge(candidate_counts,on=['round','Label'],how='left',validate='many_to_one')
link_qc['metadata_candidates']=link_qc['metadata_candidates'].fillna(0).astype(int)
link_qc['status']=np.select([link_qc['metadata_candidates']==0,link_qc['metadata_candidates']==1],
                             ['NO_MATCH','UNIQUE_MATCH'],default='AMBIGUOUS')
print('Round-aware matching:')
print(link_qc['status'].value_counts().to_dict())
print('Examples of unmatched/ambiguous:')
display(link_qc.loc[link_qc['status']!='UNIQUE_MATCH'].head(30))
# 다른 Round로 연결되는 Label을 별도로 조사해 라운드 레이블 오류 가능성 평가
all_meta_labels=set(medium['Label'].dropna().astype(str))
link_qc['label_exists_in_metadata_any_round']=link_qc['Label'].isin(all_meta_labels)
link_qc.to_csv(OUT_DIR/'curve_to_metadata_link_qc.csv',index=False,encoding='utf-8-sig')

## 6. 전역 Label 매칭 및 Round 불일치 원인 추적
라운드 표기가 이상한 경우를 위해 `Label` 단독 연결도 별도 진단합니다. 단, **전역 Label이 중복되면 무조건 모호합니다.** 두 방식 모두 불일치를 보이면 조인을 중단하고 원본 제작 규칙을 확인해야 합니다.

In [ ]:
# 6. 전역 라벨의 유일성, 누락 및 라운드 일치 여부 (진단용)
raw_count=curve_inventory.groupby('Label').size().rename('raw_occurrences')
metadata_count=medium.groupby('Label').size().rename('metadata_occurrences')
label_counts=pd.concat([raw_count,metadata_count],axis=1).fillna(0).astype(int).reset_index()
label_counts['safe_global_key']=(label_counts['raw_occurrences']==1)&(label_counts['metadata_occurrences']==1)
print('Labels unique in both:',int(label_counts['safe_global_key'].sum()))
print('Raw-only:',int(((label_counts.raw_occurrences>0)&(label_counts.metadata_occurrences==0)).sum()))
print('Metadata-only:',int(((label_counts.raw_occurrences==0)&(label_counts.metadata_occurrences>0)).sum()))
display(label_counts.loc[~label_counts.safe_global_key].head(30))
label_counts.to_csv(OUT_DIR/'global_label_audit.csv',index=False,encoding='utf-8-sig')

## 7. `K`, `r` 및 품질 코드 분포 검사
기존 평가표의 지표를 읽되 `K_info`, `r_info`의 의미는 원 논문·메타데이터 설명 없이 임의로 정의하지 않습니다.

In [ ]:
# 7. 원본 평가 필드의 결측·코드 빈도
quality=[]
for col in ['K','r','K_info','r_info']:
    s=evaluation[col]
    quality.append({'column':col,'nonmissing':int(s.notna().sum()),
                    'unique':int(s.nunique(dropna=True)),
                    'values_preview':str(s.dropna().unique()[:10].tolist())})
quality_audit=pd.DataFrame(quality)
display(quality_audit)
for col in ['K_info','r_info']:
    print(col, evaluation[col].value_counts(dropna=False).sort_index().to_dict())
quality_audit.to_csv(OUT_DIR/'evaluation_field_audit.csv',index=False,encoding='utf-8-sig')

## 8. 진단 결과에 따라 연결 가능 여부 판단
**엄격한 승인 조건:** 전체 Curve가 해당 Round에서 유일하게 매칭되고, 원본 Label이 전역 유일하며, 메타데이터 두 표의 Assay·Condition 대응이 완전히 일치해야 합니다. 승인 조건을 통과하지 않으면 결합 데이터를 생성하지 않습니다. 승인 시에도 검증된 **명시적 키 조인**만 허용합니다.

In [ ]:
# 8. 연결 승인 게이트와 검증 보고서
all_round_unique=bool((link_qc['status']=='UNIQUE_MATCH').all())
all_global_unique=bool(label_counts['safe_global_key'].all())
all_round_counts_ok=bool((round_audit['observed_count']==round_audit['expected_count']).all())
all_times_ok=bool(round_audit['time_strictly_increasing'].all())
all_meta_ok=bool(matched.all() and meta_assay_equal and meta_cond_equal)
join_validated=all([all_round_unique,all_global_unique,all_round_counts_ok,all_times_ok,all_meta_ok])
report={'dataset':'Aida–Ying BW25113','raw_curves':int(len(curve_inventory)),
 'round_counts_ok':all_round_counts_ok,'time_axes_ok':all_times_ok,
 'metadata_consistent':all_meta_ok,
 'globally_duplicated_curve_labels':int(duplicates.Label.nunique()),
 'round3_round4_overlap_labels':int(len(round3&round4)),
 'round_aware_status_counts':link_qc.status.value_counts().to_dict(),
 'global_unique_labels':all_global_unique,'all_round_matches_unique':all_round_unique,
 'join_validated':join_validated,'scientific_analysis_ready':join_validated,
 'caveat':'문자열·Round 키 일치가 성립해도 원 자료 생성 규칙 및 K/r 해석 검증은 별도 과제임'}
print(json.dumps(report,ensure_ascii=False,indent=2))
(OUT_DIR/'link_audit_report_v2.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
if join_validated:
    # 승인된 경우에만 키 테이블 저장; 원본 OD 시계열과 배지 전체의 결합은 B06-02에서 수행
    mapping=(curve_keys.merge(medium_keys,on=['round','Label'],how='left',validate='one_to_one')
       .merge(evaluation[['Label','Assay ID','Condition ID']],on=['Label','Assay ID','Condition ID'],validate='one_to_one'))
    assert len(mapping)==len(curve_inventory)
    mapping.to_csv(OUT_DIR/'VALIDATED_curve_assay_condition_mapping.csv',index=False,encoding='utf-8-sig')
    print('Validated key mapping saved:',len(mapping))
else:
    print('연결 승인 보류: 진단 CSV를 검토하고 실제 명명 규칙을 확인하세요. 결합 데이터는 생성되지 않습니다.')

## 9. 실행 후 확인할 사항 및 연습문제

핵심 출력: `duplicate_curve_labels.csv`, `curve_to_metadata_link_qc.csv`, `global_label_audit.csv`, `link_audit_report_v2.json`.

**해석의 우선순위:** ① 원본 Curve 이름 전역 중복 → ② Round 03/04 겹침 → ③ 메타데이터 Assay Round와의 매칭 → ④ K/r 의미 검토. 오류나 애매한 결과를 임의 보정하지 않습니다.

**연습문제:** 1) `Round03∩Round04`의 중복 ID는 몇 개인가? 2) 원본 곡선과 메타데이터의 전역 Label 집합 차이는 무엇인가? 3) `Assay ID`를 숫자로 변환하는 검사가 왜 부적절한가? 4) Round를 포함한 키가 필요한 상황을 설명하라. 모범답안은 Word 실습 해설서에 수록합니다.